In [1]:
import csv
import pandas as pd

GRID = "/Users/mac/Downloads/positives_grid_1h_2h.csv"
PARSE = "Thunderbird_parsed.csv"
OUT = "grid_with_content.csv"

grid = pd.read_csv(GRID)
hc = grid[grid.is_hardcase == True].copy()
wanted = set(zip(hc.timestamp.astype(str), hc.node.astype(str), hc.alert_tag.astype(str)))
print(f"grid 원인 줄: {len(hc):,}개  (열쇠 {len(wanted):,}개)")

found = {}
with open(PARSE, encoding="utf-8") as f:
    r = csv.DictReader(f)
    for i, row in enumerate(r):
        k = (row["timestamp"], row["user"], row["label"])
        if k in wanted and k not in found:
            found[k] = row["content"]
        if i % 20_000_000 == 0:
            print(f" 파싱 {i:,} 줄 훑음... (수집 {len(found):,}/{len(wanted):,})")
        if len(found) == len(wanted):
            print(" 전부 매칭 완료, 조기 종료")
            break

hc["content"] = [found.get((str(t), str(n), str(a)), "")
                 for t,n,a in zip(hc.timestamp, hc.node, hc.alert_tag)]
hc.to_csv(OUT, index = False)

matched = (hc.content != "").sum()
print(f"\n완료: {matched:,}/{len(hc):,} 줄에 원문 붙음 -> {OUT}")

if matched < len(hc):
    print(f"  못 붙은 {len(hc)-matched:,}줄: 파싱 파일에 해당 (시각·노드·alert) 조합이 없음 → 나중에 확인")


grid 원인 줄: 14,819개  (열쇠 10,889개)
 파싱 0 줄 훑음... (수집 0/10,889)
 파싱 20,000,000 줄 훑음... (수집 116/10,889)
 파싱 40,000,000 줄 훑음... (수집 381/10,889)
 파싱 60,000,000 줄 훑음... (수집 1,244/10,889)
 파싱 80,000,000 줄 훑음... (수집 1,667/10,889)
 파싱 100,000,000 줄 훑음... (수집 3,865/10,889)
 파싱 120,000,000 줄 훑음... (수집 4,808/10,889)
 파싱 140,000,000 줄 훑음... (수집 7,104/10,889)
 파싱 160,000,000 줄 훑음... (수집 7,311/10,889)
 파싱 180,000,000 줄 훑음... (수집 10,395/10,889)
 파싱 200,000,000 줄 훑음... (수집 10,696/10,889)
 전부 매칭 완료, 조기 종료

완료: 14,818/14,819 줄에 원문 붙음 -> grid_with_content.csv
  못 붙은 1줄: 파싱 파일에 해당 (시각·노드·alert) 조합이 없음 → 나중에 확인
